In [5]:
from future_rag.indexing.embeddings import EmbeddingFactory
from future_rag.indexing.indexer import IndexBuilder
from future_rag.indexing.loader import IndexLoader
from future_rag.indexing.vectorstore import VectorStore
from future_rag.knowledge.chunker import DocumentChunker
from future_rag.knowledge.loaders import DocumentLoader

from future_rag.retrieval.retriever import Retriever
from future_rag.config.settings import RAGConfig

from pathlib import Path
from dotenv import load_dotenv
import chromadb
import os
import warnings

load_dotenv()
config = RAGConfig()
PROJECT_ROOT = Path.cwd().parent

warnings.filterwarnings("ignore")

In [6]:
print("=" * 60)
print("Testing FUTURE-RAG Pipeline")
print("=" * 60)

# ------------------------------------------------------------
# 1. Load source documents
#    DocumentLoader converts PDF files into LlamaIndex Documents
# ------------------------------------------------------------
pdf_path = PROJECT_ROOT / config.documents_path
print(f"Loading Documents from: {pdf_path}")
documents = DocumentLoader.load(pdf_path)

print(f"Loaded Documents : {len(documents)}")


# ------------------------------------------------------------
# 2. Split documents into chunks (Nodes)
#    Nodes are the units stored in the vector database.
# ------------------------------------------------------------
# Chunk size and overlap are configurable parameters.
chunk_size=config.chunk_size
chunk_overlap=config.chunk_overlap

print(f"Chunk Size       : {chunk_size}")
print(f"Chunk Overlap    : {chunk_overlap}")

chunker = DocumentChunker(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)

nodes = chunker.split(documents)

print(f"Generated Nodes  : {len(nodes)}")


# ------------------------------------------------------------
# 3. Create embedding model
#    The same embedding model must be used during:
#       - indexing (creating vectors)
#       - retrieval (querying vectors)
# ------------------------------------------------------------
embedding_model = EmbeddingFactory.create(config.embedding_model)

print(
    f"Embedding Model  : {type(embedding_model).__name__}"
)


# ------------------------------------------------------------
# 4. Create / reset ChromaDB vector store
#    VectorStore is a wrapper around ChromaDB.
#    Before indexing, remove the existing collection to avoid
#    duplicated embeddings from previous test runs.
#    connect() returns the LlamaIndex ChromaVectorStore object
#    required by VectorStoreIndex.
# ------------------------------------------------------------



db_path = PROJECT_ROOT / config.vectorstore_path
collection_name = config.collection_name

print(f"Preparing ChromaDB:")
print(f"  Path       : {db_path}")
print(f"  Collection : {collection_name}")


# Delete existing collection if it exists
client = chromadb.PersistentClient(
    path=db_path
)

existing_collections = [
    collection.name
    for collection in client.list_collections()
]

if collection_name in existing_collections:
    client.delete_collection(collection_name)
    print(f"Deleted existing collection: {collection_name}")
else:
    print("No existing collection found")


# Create a fresh vector store
vector_store_manager = VectorStore(
    db_path=db_path,
    collection_name=collection_name,
)

chroma_vector_store = vector_store_manager.connect()

print(
    f"Vector Store    : {type(chroma_vector_store).__name__}"
)


# ------------------------------------------------------------
# 5. Build VectorStoreIndex
#    IndexBuilder combines:
#       - Nodes
#       - Embedding model
#       - Vector store
# ------------------------------------------------------------
builder = IndexBuilder(
    embedding_model=embedding_model,
    vector_store=chroma_vector_store,
)

index = builder.build(nodes)


# ------------------------------------------------------------
# 6. Validate result
# ------------------------------------------------------------
print(f"Index Type      : {type(index).__name__}")


# ------------------------------------------------------------
# 7. Test Retrieval Pipeline
#    Verify that the index can retrieve relevant chunks
#    and inspect similarity scores returned by LlamaIndex.
# ------------------------------------------------------------


# Create retriever
# similarity_top_k defines the number of relevant chunks returned
similarity_top_k = config.similarity_top_k
retriever = Retriever(
    index,
    similarity_top_k=similarity_top_k,
)

#Questions
#What is the main purpose of the FUTURE-AI framework?
#What are the six guiding principles of the FUTURE-AI framework?
#Why does the paper argue that AI adoption in healthcare has remained limited despite advances in AI research?
#How many experts and countries were involved in developing the FUTURE-AI guidelines, and how was consensus achieved?
#What is the difference between the principles of fairness and universality in the FUTURE-AI framework?
#Why does the framework recommend evaluating AI systems using external datasets and multiple clinical sites?
#According to the paper, what are the main responsibilities included under the principle of traceability throughout an AI tool's lifecycle?
#How does the FUTURE-AI framework distinguish between AI tools developed for research purposes and those intended for clinical deployment?
#Describe how the four phases of the AI lifecycle (design, development, validation, and deployment) are connected to the implementation of the FUTURE-AI recommendations.
#The paper emphasizes that trustworthy AI requires more than technical performance. Using the FUTURE-AI framework, explain how fairness, usability, robustness, explainability, traceability, universality, and the general recommendations work together to support safe and deployable healthcare AI.


# User query
query = "What are the six guiding principles of the FUTURE-AI framework?"

print(f"\nQuery: {query}")

# Retrieve relevant nodes from the vector store
results = retriever.retrieve(query)

print(f"\nRetrieved Chunks : {len(results)}")

# ------------------------------------------------------------
# Analyze retrieval similarity scores
# LlamaIndex returns NodeWithScore objects.
# The score indicates how similar each chunk is to the query.
# Higher scores generally represent better matches.
# ------------------------------------------------------------

if results:
    min_score = min(node.score for node in results)
    max_score = max(node.score for node in results)

    print(f"Minimum Similarity Score : {min_score:.4f}")
    print(f"Maximum Similarity Score : {max_score:.4f}")


# ------------------------------------------------------------
# Display retrieved chunks
# ------------------------------------------------------------

for i, node_with_score in enumerate(results, start=1):

    print("\n" + "=" * 60)
    print(f"Chunk {i}")
    print("=" * 60)

    # Similarity score from LlamaIndex
    print(f"Similarity Score : {node_with_score.score:.4f}")

    # Retrieved text content
    print("\nText:")
    print(node_with_score.node.text[:500])

    # Optional metadata information
    print("\nMetadata:")
    print(node_with_score.node.metadata)


print("\n✅ Pipeline test passed!")
    

Testing FUTURE-RAG Pipeline
Loading Documents from: d:\Future AI Workspace\future-rag\data\knowledge\pdf
Loaded Documents : 22
Chunk Size       : 1024
Chunk Overlap    : 256
Generated Nodes  : 60


2026-07-29 19:44:42,652 - INFO - Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
2026-07-29 19:44:45,315 - INFO - Loaded 1 prompt with these keys: ['query']


Embedding Model  : HuggingFaceEmbedding
Preparing ChromaDB:
  Path       : d:\Future AI Workspace\future-rag\data\vectorstores\chroma
  Collection : future_rag
Deleted existing collection: future_rag
Vector Store    : ChromaVectorStore
Index Type      : VectorStoreIndex

Query: What are the six guiding principles of the FUTURE-AI framework?

Retrieved Chunks : 3
Minimum Similarity Score : 0.6235
Maximum Similarity Score : 0.6859

Chunk 1
Similarity Score : 0.6859

Text:
RESEARCH METHODS AND REPORTINGRESEARCH METHODS AND REPORTING
mobile health, electronic health records, surgery, 
pathology). To develop the FUTURE-AI framework, 
we drew inspiration from the FAIR principles for data 
management, and defined concise recommendations 
organised according to six guiding principles—fairness, 
universality, traceability, usability, robustness, and 
explainability (fig 2).
Methods
FUTURE-AI is a structured framework that provides 
guiding principles and step-by-step re

Metadata:
{'page_label'